# HW7 — Валидация, утечки и неопределённость

> **Начинающему трейдеру — начать отсюда.** Раздел «Справочник» объясняет простыми словами все
> термины валидации, которые дальше используются. Основной текст показывает главное правило
> профессии: **бэктест врёт, если вы не сделали ничего, чтобы его поймать.** Ноутбук не обучает
> модель — он всеми силами пытается **опровергнуть** результаты.

## Справочник для начинающего алгоритмического трейдера

| Термин | Что это простыми словами |
|---|---|
| **Валидация (CV, cross-validation)** | Разбиение данных на части так, чтобы проверить модель на той части, которой она не училась |
| **K-fold** | Данные на K частей; по очереди каждая часть — тест, остальные — обучение |
| **Walk-forward** | «Живой» режим: учимся только на прошлом, проверяем на следующем куске, двигаем окно вправо. Единственный способ, имитирующий реальную торговлю |
| **Purged K-fold (очищенная)** | K-fold, из которого **выкинуты** наблюдения, чья метка пересекается с тестовым окном |
| **Embargo (эмбарго)** | «Карантин»: полоса баров вокруг тестового блока, которая выбрасывается из обучения целиком |
| **Leakage (утечка)** | Модель при обучении или оценке увидела информацию из будущего. Результат перестаёт значить что-либо |
| **Scaler fit на train+test** | Самая частая утечка: статистика масштабирования (среднее/сигма) посчитана с учётом теста → тест «просочился» в обучение |
| **Forward-looking feature** | Признак, значение которого известно только в будущем. Подача его модели = подглядывание в ответ |
| **Deadband (мёртвая зона)** | Допуск ±ε вокруг нуля, где модель говорит «держаться», а не «вверх/вниз». Убирает шум стакана заявок |
| **Majority-class benchmark** | Точность тривиального классификатора «всегда предсказывать самый частый класс» |
| **Buy-and-hold benchmark** | Доходность простого «купил и держишь». Стратегия обязана её обойти |
| **Sharpe / Sortino / CAGR** | Доходность на единицу риска (по волатильности / по просадке) и среднегодовой темп роста |
| **CPCV (combinatorial purged CV)** | Всевые комбинации тестовых групп → много вариантов бэктеста сразу → распределение результатов |
| **PSR / DSR** | Вероятность того, что Sharpe ≠ 0 / ≠ «Sharpe лучшего из N попыток, полученный на удачу» |
| **MinTRL (track record)** | Сколько лет наблюдений нужно, чтобы подтвердить заявленный Sharpe вообще |
| **Spearman rank correlation** | Корреляция *ранжировок* (не значений): предсказывает ли CV-рейтинг OOS-рейтинг |
| **Selection bias** | Выбрали лучший из N — и его результат систематически завышен |

## Зачем существует этот ноутбук

`HW6` дал неудобный результат: зооподбор ARIMA, ETS и state-space обогнал случайное блуждание
на доходностях SPY с большим отрывом, и **ни одна из них** не обогнала константу. Отрыв был
реален, вывод пуст, потому что бейзлайн был реализован неверно.

Это типичный способ провала всей области, и он заслуживает ноутбука, который атакует его напрямую,
а не ещё одну модель. В `HW1`–`HW5` нет ни валидации, ни контроля утечек, ни неопределённости
бэктеста — ни в одной форме, которую можно было бы защитить на код-ревью.
Поэтому **этот ноутбук не обучает модель для прогноза.** Он всеми силами пытается **опровергнуть**
результаты, используя одни и те же данные и одни и те же модели на протяжении всего — так что
меняется *только* дисциплина валидации.

## Пять вопросов

Каждый отвечается экспериментом, где модель фиксируется, а меняется только метод:

| # | Вопрос | Эксперимент |
|---|---|---|
| 1 | Меняет ли **правило разбиения** ответ? | случайный K-fold vs очищенный K-fold vs walk-forward |
| 2 | На сколько утечка в предобработке раздувает число? | скейлер по фолдам vs скейлер, обученный на train+test |
| 3 | Можно ли «накрутить» метрику без аудита причинных признаков? | один признак, смотрящий вперёд, добавлен намеренно |
| 4 | Сколько в «лучшей модели» — шум отбора? | 24 конфигурации, отбор на CV, оценка вне выборки |
| 5 | Насколько ответ зависит от **того, где стояло тестовое окно**? | распределение путей CPCV |

## Стандарт, которому ноутбук сам себя обязывает

Три правила, выстраданные в `HW6`:

* **Каждая метрика паруется с нулевым бенчмарком.** Точность бессмысленна рядом с точностью
  большинства классов; Sharpe бессмысленен рядом с buy-and-hold; откалиброванный интервал
  бессмысленен, если не печатается частота промахов.
* **Ничего не выбирается на финальном тестовом окне.** Последние 20% баров трогаются один раз —
  той самой одной моделью, которую выбрала честная процедура, — и далее используются как
  измерительный прибор.
* **Неопределённость печатается, а не подразумевается.** Точечная оценка Sharpe из нескольких
  сотен баров — утверждение о шуме. Каждая заглавная цифра здесь идёт с интервалом.

Там, где честный ответ — «edge нет», ноутбук так и пишет. **Отрицательный результат, чётко
сформулированный, ценнее для исследовательской команды, чем положительный, который
распадается при проверке.**

In [1]:
import json
import os
import sys
import warnings
from itertools import product

import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd()
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

from mlfin_core import (
    TRADING_DAYS,
    FoldSafePreprocessor,
    annualised_return,
    apply_causal_execution,
    bootstrap_sharpe_ci,
    buy_and_hold_returns,
    cpcv_backtest_paths,
    combinatorial_purged_cv_splits,
    deflated_sharpe_ratio,
    load_ohlcv,
    make_direction_target,
    make_feature_frame,
    min_track_record_length,
    positions_to_returns,
    probabilistic_sharpe_ratio,
    profit_factor,
    purged_kfold_splits,
    sharpe_ratio,
    sortino_ratio,
    walk_forward_splits,
    win_rate,
)
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold

print("mlfin_core загружен | TRADING_DAYS =", TRADING_DAYS)

mlfin_core загружен | TRADING_DAYS = 252


## Конфигурация

Два выбора требуют собственного оправдания, и оба — те, что чаще всего нарушались в более ранних
ноутбуках этого репозитория.

**Мёртвая зона (deadband) на цели.** `make_direction_target` поддерживает `deadband`: доходность
вперёд внутри ±`DEAD_BAND` становится третьим классом *«держаться»*, а не принудительно
размечается как вверх или вниз. Без неё цель — «дрогнула ли цена вверх», что в основном бегство
спреда и микроструктурный шум, и модель может набрать на этом хорошую оценку, не имея чего
торговать.

**Выделенное финальное окно.** Последние `OOS_FRACTION` баров никогда не используются для отбора
моделей, статистик предобработки и ранней остановки. Дополнительно выбрасывается `EMBARGO`
баров вокруг тестовых блоков, потому что метка, окно которой пересекает границу теста, уже
«видела» тестовый период.

**Почему `REFIT_EVERY` совпадает с `HW6`.** Переобучение на каждом баре в принципе чище, но на
практике недоступно по деньгам; переобучение раз в неделю с оценкой *всегда до* следующего
переобучения сохраняет причинность симуляции.

In [2]:
TICKER = "SPY"
START = "2015-01-01"
DATA_END = "2026-10-05"   # зафиксировано: опубликованные числа обязаны воспроизводиться
DATA_DIR = os.path.join(NOTEBOOK_DIR, "data")   # закоммиченный снимок: читается вместо запроса к API
HORIZON = 1              # горизонт метки в момент решения, в барах
DEAD_BAND = 0.0005       # ±5 bp превращается в класс «держаться»
EMBARGO = 5              # бары, отбрасываемые по обе стороны тестового блока
OOS_FRACTION = 0.20      # финальное окно: трогается ровно один раз
REFIT_EVERY = 21         # периодичность переобучения в walk-forward
COST_BPS = 10.0          # предполагаемая издержка за полный круг
N_SPLITS = 5
SR_TARGET = 1.0          # годовой Sharpe, с которым сравнивается история торговли
RNG_SEED = 0

df, source = load_ohlcv(TICKER, START, end=DATA_END, local_dir=DATA_DIR, allow_synthetic=True)
print(f"{TICKER}: {len(df)} баров | {df.index[0].date()} -> {df.index[-1].date()}")
print(f"источник данных: {source}")
if "synthetic" in str(source).lower():
    print("ВНИМАНИЕ: используется синтетическая заглушка. Каждое число ниже — свойство")
    print("генератора, а не рынка. Выводы нужно читать как демонстрацию метода.")

SPY: 3066 баров | 2015-01-02 -> 2026-10-02
источник данных: SPY (local: C:\Users\MV\AppData\Local\Temp\opencode\ML_Fin_Notebooks\data\SPY.csv)


## Признаки, цель и контракт момента решения

**Что гарантирует контракт.** `make_feature_frame` возвращает столбцы, которые являются
функциями цены закрытия бара *t* и более ранних. `make_direction_target` возвращает метку для
бара *t*, которая наблюдаема лишь в *t + HORIZON*. **Этот разрыв и есть вся причина
существования purging и embargo,** и следующая ячейка это измеряет, а не утверждает.

**Отдельно фиксируется частота класса большинства** — на обучающей доле. Она становится
бенчмарком точности во всех экспериментах ниже: **урок `HW6`, применённый к классификации.**

In [3]:
feats = make_feature_frame(df, n_lags=5, extra_indicators=True)
y = make_direction_target(df["Close"], horizon=HORIZON, deadband=DEAD_BAND, labels=(0, 1, 2))
y = y.rename("y")

data = feats.join(y, how="inner").dropna(how="all")
data = data[data["y"].notna()].copy()
close = df["Close"].reindex(data.index)

n_obs = len(data)
n_oos = int(round(n_obs * OOS_FRACTION))
IS_SLICE = slice(0, n_obs - n_oos)
OOS_SLICE = slice(n_obs - n_oos, n_obs)

X = data.drop(columns=["y"])
y = data["y"]
X_IS, y_IS = X.iloc[IS_SLICE], y.iloc[IS_SLICE]
X_OOS, y_OOS = X.iloc[OOS_SLICE], y.iloc[OOS_SLICE]
close_IS, close_OOS = close.iloc[IS_SLICE], close.iloc[OOS_SLICE]

CLASS_NAMES = {0: "sell", 1: "buy", 2: "hold"}
print(f"признаки        : {X.shape[1]} столбцов, {X.shape[0]} строк")
print(f"окно признаков  : {X.index[0].date()} -> {X.index[-1].date()}")
print(f"IS   (отбор)    : {len(X_IS)} баров  {X_IS.index[0].date()} -> {X_IS.index[-1].date()}")
print(f"OOS  (один раз) : {len(X_OOS)} баров  {X_OOS.index[0].date()} -> {X_OOS.index[-1].date()}")

print("\nclass balance")
bal = pd.DataFrame({
    "IS_share": y_IS.value_counts(normalize=True).sort_index(),
    "OOS_share": y_OOS.value_counts(normalize=True).sort_index(),
})
bal.index = [CLASS_NAMES[i] for i in bal.index]
print(bal.round(4).to_string())

# Бенчмарк. Обучен только на IS и применён к OOS, ни разу не глядя на него.
MAJORITY_CLASS = int(y_IS.value_counts().idxmax())
CLIMATOLOGY_ACC = float((y_OOS == MAJORITY_CLASS).mean())
print(f"\nкласс большинства на IS     : {CLASS_NAMES[MAJORITY_CLASS]}")
print(f"точность OOS константы «всегда покупать» : {CLIMATOLOGY_ACC:.4f}")
print("Любой классификатор ниже сравнивается с ЭТИМ числом, а не с 1/3 по умолчанию,")
print("потому что три класса встречаются с разной частотой.")

признаки        : 23 столбцов, 3065 строк
окно признаков  : 2015-01-02 -> 2026-10-01
IS   (отбор)    : 2452 баров  2015-01-02 -> 2024-05-27
OOS  (один раз) : 613 баров  2024-05-28 -> 2026-10-01

class balance
      IS_share  OOS_share
sell    0.4091     0.3834
buy     0.4898     0.5073
hold    0.1011     0.1093

класс большинства на IS     : buy
точность OOS константы «всегда покупать» : 0.5073
Любой классификатор ниже сравнивается с ЭТИМ числом, а не с 1/3 по умолчанию,
потому что три класса встречаются с разной частотой.


## Почему purging и embargo необязательны к опущению

**Суть утечки в одном абзаце.** Метка бара *t* реализуется в *t + HORIZON*. Наблюдение на
обучении в момент *t* несёт информацию о тестовом окне всякий раз, когда *t* находится в
`HORIZON` барах от него — **даже если *t* строго *до* тестового блока и проходит наивную
проверку временного порядка.**

**Что делает ячейка ниже.** Берётся непрерывный тестовый блок, и задаётся вопрос: сколько
обучающих строк оставляет каждая схема и сколько из них имеют окно метки, пересекающее
тестовый блок. Строка, выжившая при обычном хронологическом разбиении, но чья метка
пересекается с тестом, — **утечка, которая проходит фильтр по дате.** Именно так выглядят
баги, которые находят только на проде.

In [4]:
demo_n = 1200
demo_purged = purged_kfold_splits(demo_n, n_splits=5, embargo=EMBARGO, t1=HORIZON)[2]
demo_test = np.asarray(demo_purged[1])
demo_train_naive = np.arange(0, int(demo_test[0]))

# Обе схемы оцениваются на ОДНОМ И ТОМ ЖЕ тестовом блоке, поэтому единственное отличие —
# выбрасываются ли строки, окно метки которых дотягивается до этого блока.
demo_splits = [
    ("наивное хронологическое", (demo_train_naive, demo_test)),
    ("очищенное + embargo", (np.asarray(demo_purged[0]), demo_test)),
]

rows = []
for label, (tr, te) in demo_splits:
    tr = np.asarray(tr)
    te = np.asarray(te)
    t0, t1 = int(te[0]), int(te[-1])
    # Обучающее окно метки [j, j + HORIZON] касается тестового блока, если оно доходит до t0.
    overlaps = int(((tr + HORIZON) >= t0).sum())
    inside = int(((tr >= t0) & (tr <= t1)).sum())
    rows.append({
        "scheme": label,
        "train_rows": len(tr),
        "rows_inside_test": inside,
        "label_overlaps_test": overlaps,
        "leaked_rows_pct": 100.0 * overlaps / max(len(tr), 1),
    })

leak_tbl = pd.DataFrame(rows)
print(f"тестовый блок = бары {t0}..{t1} из {demo_n}, горизонт метки = {HORIZON}\n")
print(leak_tbl.round(3).to_string(index=False))
print(
    "\nНаивное разбиение оставляет обучающие строки, чьи метки решаются внутри тестового окна.\n"
    "Хронологически они 'до' тестового набора, поэтому фильтр по дате их пропускает.\n"
    f"Очищение их убирает, а embargo в {EMBARGO} баров дополнительно отбрасывает бары сразу за\n"
    "блоком, чьи признаки ещё успокаиваются после экстремумов самого тестового периода."
)

тестовый блок = бары 480..719 из 1200, горизонт метки = 1

                 scheme  train_rows  rows_inside_test  label_overlaps_test  leaked_rows_pct
наивное хронологическое         480                 0                    1            0.208
    очищенное + embargo         954                 0                  475           49.790

Наивное разбиение оставляет обучающие строки, чьи метки решаются внутри тестового окна.
Хронологически они 'до' тестового набора, поэтому фильтр по дате их пропускает.
Очищение их убирает, а embargo в 5 баров дополнительно отбрасывает бары сразу за
блоком, чьи признаки ещё успокаиваются после экстремумов самого тестового периода.


In [5]:
# -- общие вспомогательные функции эксперимента ------------------------------- #
# Фиксированный порядок столбцов в векторах вероятностей 3 классов, чтобы фолды с разными
# множествами классов оставались сопоставимыми при их склейке.
CLASS_ORDER = [0, 1, 2]


def fit_predict_scores(X_tr, y_tr, X_te, y_te, model, leak: bool = False):
    """Точность на одном фолде, с дисциплиной предобработки, выбранной параметром `leak`.

    leak=False : статистика только из обучающего фолда       (правильно)
    leak=True  : статистика из train + test совместно         (антипаттерн)
    """
    if leak:
        Xtr_s, Xte_s = FoldSafePreprocessor.fit_on_full_sample(X_tr, X_te)
    else:
        pre = FoldSafePreprocessor().fit(X_tr)
        Xtr_s, Xte_s = pre.transform(X_tr), pre.transform(X_te)
    model.fit(Xtr_s, y_tr.to_numpy())
    pred = model.predict(Xte_s)
    return float((pred == y_te.to_numpy()).mean())


def cv_mean_accuracy(splits, Xd, yd, make_model, leak: bool = False) -> float:
    return float(np.mean([
        fit_predict_scores(Xd.iloc[tr], yd.iloc[tr], Xd.iloc[te], yd.iloc[te], make_model(), leak=leak)
        for tr, te in splits
    ]))


def evaluate_on_oos(C_value, leak: bool = False, seed: int = RNG_SEED) -> dict:
    """Обучение на всём окне IS и однократное предсказание нетронутого финального окна."""
    model = LogisticRegression(C=C_value, max_iter=2000, random_state=seed)
    if leak:
        Xtr_s, Xte_s = FoldSafePreprocessor.fit_on_full_sample(X_IS, X_OOS)
    else:
        pre = FoldSafePreprocessor().fit(X_IS)
        Xtr_s, Xte_s = pre.transform(X_IS), pre.transform(X_OOS)
    model.fit(Xtr_s, y_IS.to_numpy())
    pred = model.predict(Xte_s)
    acc = float((pred == y_OOS.to_numpy()).mean())
    return {"oos_accuracy": acc, "skill_vs_climatology": acc - CLIMATOLOGY_ACC}


def leakage_audit(Xd: pd.DataFrame, yd: pd.Series, rel_multiple: float = 5.0,
                  abs_threshold: float = 0.5) -> pd.DataFrame:
    """Отметить признаки, которые ведут себя как переформулировка цели.

    Для каждого признака и каждого класса считается точечно-бисериальная
    корреляция между признаком и бинарным индикатором этого класса, затем
    берётся наибольшее по модулю значение.

    Абсолютного порога здесь недостаточно, и это стоит пояснить: у цели есть
    класс `hold`, поэтому даже признак, который *и есть* доходность вперёд,
    не может иметь корреляцию 1.0 с каким-либо индикатором класса — deadband
    намеренно ломает монотонную связь. Поэтому детектор использует два критерия:

    * ``rel_multiple``: корреляция этого признака во много раз больше медианной
      корреляции по всему набору признаков? Это не зависит от шкалы и не
      предполагает, что цель является чистой переформулировкой чего-либо.
    * ``abs_threshold``: высокая абсолютная корреляция — для чистого
      бинарного случая.

    Оба критерия выводятся, а ячейка ниже показывает, какой из них сработал.
    """
    rows = []
    yv = yd.to_numpy()
    for col in Xd.columns:
        v = pd.to_numeric(Xd[col], errors="coerce")
        best, best_k = np.nan, None
        for k in np.unique(yv):
            ind = (yv == k).astype(float)
            ok = v.notna()
            if ok.sum() < 30 or ind[ok].std() == 0 or v[ok].std() == 0:
                continue
            r = float(np.corrcoef(v[ok], ind[ok])[0, 1])
            if np.isfinite(r) and (not np.isfinite(best) or abs(r) > abs(best)):
                best, best_k = r, int(k)
        rows.append({"feature": col, "max_biserial_r": best, "vs_class": best_k})

    out = pd.DataFrame(rows)
    out["abs_r"] = out["max_biserial_r"].abs()
    med = float(out["abs_r"].median())
    out["ratio_to_median"] = out["abs_r"] / med if med > 0 else np.nan
    out["flagged"] = (out["ratio_to_median"] >= rel_multiple) | (out["abs_r"] >= abs_threshold)
    out.attrs["median_abs_r"] = med
    return out.sort_values("abs_r", ascending=False).reset_index(drop=True)


print("helpers defined")

helpers defined


# Эксперимент 1 — меняет ли правило разбиения ответ?

Одна и та же модель, одни и те же признаки, одно и то же финальное тестовое окно. **Меняется
только правило выбора силы регуляризации:**

* **Случайный K-fold** — дефолт в большинстве туториалов, включая большинство прикладного ML-кода.
* **Очищенный K-fold с embargo** — конструкция Лопеса де Прадо, реализованная и покрыта
  юнит-тестами в `mlfin_core.purged_kfold_splits`.
* **Walk-forward** — расширяющееся обучающее окно, один непрерывный тестовый блок на фолд.

Каждая схема выбирает разный `C`, и каждая выбранная модель затем оценивается **один раз** на
финальном окне OOS. **Если правило разбиения — техническая деталь, три ответа должны
совпасть. Они не совпадают, и величина расхождения — и есть находка.**

In [6]:
C_GRID = [0.01, 0.05, 0.1, 0.5, 1.0, 5.0]

# walk_forward_splits требует min_train_size + n_splits*(test_size + embargo) <= n_obs,
# поэтому размер тестового блока задаётся по фактически доступному, а не захардкожен.
WF_MIN_TRAIN = max(60, len(X_IS) // 3)
WF_TEST = max(1, (len(X_IS) - WF_MIN_TRAIN - N_SPLITS * EMBARGO) // N_SPLITS)
print(f"бюджет walk-forward: min_train={WF_MIN_TRAIN} test_size={WF_TEST} "
      f"embargo={EMBARGO} на {len(X_IS)} барах IS")

schemes = {
    "случайный K-fold": list(KFold(n_splits=N_SPLITS, shuffle=True, random_state=RNG_SEED).split(X_IS)),
    "очищенный K-fold": [(tr, te) for tr, te in purged_kfold_splits(
        len(X_IS), n_splits=N_SPLITS, embargo=EMBARGO, t1=HORIZON)],
    "walk-forward": walk_forward_splits(
        len(X_IS), n_splits=N_SPLITS, min_train_size=WF_MIN_TRAIN,
        test_size=WF_TEST, embargo=EMBARGO),
}

cv_rows, oos_rows = [], []
for scheme_name, splits in schemes.items():
    for C_value in C_GRID:
        acc = cv_mean_accuracy(splits, X_IS, y_IS, lambda C=C_value: LogisticRegression(
            C=C, max_iter=2000, random_state=RNG_SEED))
        cv_rows.append({"scheme": scheme_name, "C": C_value, "cv_accuracy": acc})

    scheme_df = pd.DataFrame([r for r in cv_rows if r["scheme"] == scheme_name])
    best = scheme_df.loc[scheme_df["cv_accuracy"].idxmax()]
    res = evaluate_on_oos(float(best["C"]))
    oos_rows.append({
        "scheme": scheme_name,
        "selected_C": float(best["C"]),
        "cv_accuracy": float(best["cv_accuracy"]),
        "oos_accuracy": res["oos_accuracy"],
        "skill_vs_climatology": res["skill_vs_climatology"],
    })

cv_tbl = pd.DataFrame(cv_rows)
oos_tbl = pd.DataFrame(oos_rows).sort_values("oos_accuracy", ascending=False)

print("Точность CV по схемам и силе регуляризации\n")
print(cv_tbl.pivot(index="C", columns="scheme", values="cv_accuracy").round(4).to_string())
print(f"\nКаждая схема выбирает свою модель, и затем эта модель оценивается один раз на нетронутом")
print(f"окне OOS. Климатология (всегда '{CLASS_NAMES[MAJORITY_CLASS]}') = {CLIMATOLOGY_ACC:.4f}\n")
print(oos_tbl.round(4).to_string(index=False))

oos_spread = float(oos_tbl["oos_accuracy"].max() - oos_tbl["oos_accuracy"].min())
cv_spread = float(cv_tbl["cv_accuracy"].max() - cv_tbl["cv_accuracy"].min())
print(f"\nразброс точности OOS по схемам        : {oos_spread:.4f} п.п. точности")
print(f"разброс точности CV по всем C         : {cv_spread:.4f} п.п. точности")
print(f"диапазон CV в {cv_spread/max(oos_spread,1e-12):.1f} раза больше расхождения между схемами")
print("\nСигнал CV меньше, чем выбор схемы валидации. Это первый")
print("тревожный сигнал: сетку, достаточно тонкую, чтобы это имело значение, такие данные не решают.")

бюджет walk-forward: min_train=817 test_size=322 embargo=5 на 2452 барах IS


Точность CV по схемам и силе регуляризации

scheme  walk-forward  очищенный K-fold  случайный K-fold
C                                                       
0.01          0.4888            0.4686            0.4886
0.05          0.4888            0.4666            0.4853
0.10          0.4894            0.4682            0.4853
0.50          0.4832            0.4666            0.4829
1.00          0.4807            0.4678            0.4841
5.00          0.4801            0.4666            0.4800

Каждая схема выбирает свою модель, и затем эта модель оценивается один раз на нетронутом
окне OOS. Климатология (всегда 'buy') = 0.5073

          scheme  selected_C  cv_accuracy  oos_accuracy  skill_vs_climatology
случайный K-fold        0.01       0.4886        0.4910               -0.0163
очищенный K-fold        0.01       0.4686        0.4910               -0.0163
    walk-forward        0.10       0.4894        0.4894               -0.0179

разброс точности OOS по схемам        : 0.0016 п.

# Эксперимент 2 — на сколько утечка в предобработке раздувает число?

**Самый частый баг этого репозитория** — обучить скейлер на `concat(X_train, X_test)` *до*
разбиения. `mlfin_core` намеренно хранит сломанный вариант как
`FoldSafePreprocessor.fit_on_full_sample` **именно затем, чтобы ущерб был измерен, а не
заявлен на словах.**

**Честное предсказание, сформулированное до просмотра:** на признаках, которые уже являются
отношениями и стандартизованными осцилляторами, эффект должен быть небольшим, и отчёт
«небольшой» полезнее, чем обычно завышенный заголовок. Если эффект окажется большим — и это
стоит знать, поэтому печатается в любом случае.

In [7]:
purged_IS = [(tr, te) for tr, te in purged_kfold_splits(
    len(X_IS), n_splits=N_SPLITS, embargo=EMBARGO, t1=HORIZON)]

leak_rows = []
for C_value in C_GRID:
    safe_cv = cv_mean_accuracy(purged_IS, X_IS, y_IS, lambda C=C_value: LogisticRegression(
        C=C, max_iter=2000, random_state=RNG_SEED), leak=False)
    leaky_cv = cv_mean_accuracy(purged_IS, X_IS, y_IS, lambda C=C_value: LogisticRegression(
        C=C, max_iter=2000, random_state=RNG_SEED), leak=True)
    safe_oos = evaluate_on_oos(C_value, leak=False)["oos_accuracy"]
    leaky_oos = evaluate_on_oos(C_value, leak=True)["oos_accuracy"]
    leak_rows.append({
        "C": C_value,
        "cv_fold_safe": safe_cv,
        "cv_leaky_scaler": leaky_cv,
        "cv_delta": leaky_cv - safe_cv,
        "oos_fold_safe": safe_oos,
        "oos_leaky_scaler": leaky_oos,
        "oos_delta": leaky_oos - safe_oos,
    })

leak_tbl = pd.DataFrame(leak_rows)
print("Утечка в предобработке: те же фолды, та же модель\n")
print(leak_tbl.round(4).to_string(index=False))

max_cv_delta = float(leak_tbl["cv_delta"].abs().max())
max_oos_delta = float(leak_tbl["oos_delta"].abs().max())
worst_C = leak_tbl.loc[leak_tbl["cv_delta"].abs().idxmax(), "C"]
print(f"\nмакс. искажение CV от течущего скейлера  : {max_cv_delta:+.4f} п.п. точности (C={worst_C})")
print(f"макс. искажение OOS от течущего скейлера: {max_oos_delta:+.4f} п.п. точности")

LEAK_EFFECT_MATERIAL = max_cv_delta > 0.005
if LEAK_EFFECT_MATERIAL:
    print("\nБаг скейлера здесь ЗНАЧИМ. Не публикуйте ни одного числа, не указав,")
    print("какая дисциплина предобработки его породила.")
else:
    print("\nБаг скейлера на ЭТОМ наборе признаков незначим, потому что большинство столбцов уже")
    print("отношения или ограниченные осцилляторы. Это свойство признаков, а не разрешение")
    print("течь: тот же код на сырых ценах или на любой нормализации, зависящей от цели,")
    print("утёк бы, и тест выше тогда провалится. Оставьте fold-safe путь по умолчанию.")
print(f"\nЧестный отчёт: максимальное наблюдаемое искажение {max_cv_delta:+.4f} в CV и")
print(f"{max_oos_delta:+.4f} на финальном окне. Не заявляется ни одно завышение, которое не измерено.")

Утечка в предобработке: те же фолды, та же модель

   C  cv_fold_safe  cv_leaky_scaler  cv_delta  oos_fold_safe  oos_leaky_scaler  oos_delta
0.01        0.4686           0.4707    0.0020         0.4910            0.4910     0.0000
0.05        0.4666           0.4666   -0.0000         0.4910            0.4910     0.0000
0.10        0.4682           0.4686    0.0004         0.4894            0.4910     0.0016
0.50        0.4666           0.4674    0.0008         0.4910            0.4878    -0.0033
1.00        0.4678           0.4670   -0.0008         0.4845            0.4829    -0.0016
5.00        0.4666           0.4666    0.0000         0.4796            0.4780    -0.0016

макс. искажение CV от течущего скейлера  : +0.0020 п.п. точности (C=0.01)
макс. искажение OOS от течущего скейлера: +0.0033 п.п. точности

Баг скейлера на ЭТОМ наборе признаков незначим, потому что большинство столбцов уже
отношения или ограниченные осцилляторы. Это свойство признаков, а не разрешение
течь: тот же ко

# Эксперимент 3 — можно ли накрутить метрику? (признак, смотрящий вперёд)

**Что добавляется.** Один признак, недоступный в момент решения, — *прямая* доходность от бара
*t* до *t + HORIZON*. Если конвейер измеряет то, что заявляет, точность должна приблизиться к
1.0 — и на CV-фолдах, и на окне OOS.

**Это не чучело.** Именно эта ошибка стоит за большинством утверждений «наша модель даёт 94%
точности» в ноутбуках розничных финансов и по крайней мере за одним сломанным ноутбуком в этом
репозитории. **Вторая половина эксперимента — то, что действительно важно:** хелпер
`leakage_audit` автоматически находит виновный столбец **по одним только данным**, без того,
чтобы кто-то помнил, что его добавили.

In [8]:
close_all = df["Close"]
leak_feature = close_all.pct_change(-HORIZON).rename("FORWARD_RETURN_LEAK")
audit_before = leakage_audit(X, y)
leaky_X = X.copy()
leaky_X["FORWARD_RETURN_LEAK"] = leak_feature.reindex(X.index)
audit_after = leakage_audit(leaky_X, y)

purged_IS_full = [(tr, te) for tr, te in purged_kfold_splits(
    len(X), n_splits=N_SPLITS, embargo=EMBARGO, t1=HORIZON)]

safe_cv = cv_mean_accuracy(purged_IS_full, X, y, lambda: LogisticRegression(C=1.0, max_iter=2000))
leaky_cv = cv_mean_accuracy(purged_IS_full, leaky_X, y, lambda: LogisticRegression(C=1.0, max_iter=2000))

print("топ-5 корреляций признак/цель ДО добавления утечки")
print(audit_before.head(5).round(4).to_string(index=False))
print("\nтоп-5 корреляций признак/цель ПОСЛЕ добавления утечки")
print(audit_after.head(5).round(4).to_string(index=False))

flagged = audit_after[audit_after["flagged"]]
strongest_causal = audit_before.iloc[0]
leak_row = audit_after.iloc[0]
print(f"\nпризнаков, автоматически отмеченных как похожих на метку : {len(flagged)}")
if len(flagged):
    print(flagged[["feature", "max_biserial_r", "ratio_to_median", "flagged"]].round(3).to_string(index=False))

print(f"\nмедиана |r| по всем признакам       : {audit_after.attrs['median_abs_r']:.4f}")
print(f"сильнейший причинный признак    : {strongest_causal['feature']} "
      f"(|r| = {strongest_causal['abs_r']:.4f})")
print(f"признак «доходность вперёд»     : {leak_row['feature']} "
      f"(|r| = {leak_row['abs_r']:.4f}, в {leak_row['ratio_to_median']:.1f} раза больше медианы)")
print(
    "\nОбратите внимание: корреляция ~0.64, а НЕ ~1.0, и это правильно, а не сбой\n"
    "детектора: deadband создаёт средний класс 'hold', поэтому ни один признак не может быть\n"
    "идеальной переформулировкой метки из 3 классов. Абсолютный порог 0.95 ПРОПУСТИЛ бы эту утечку.\n"
    "Устойчивый критерий — корреляция значительно выше медианы по всему набору признаков —\n"
    "её ловит, именно поэтому детектор использует оба."
)

print(f"\nточность CV, только причинные      : {safe_cv:.4f}")
print(f"точность CV, один признак вперёд   : {leaky_cv:.4f}")
print(f"точность климатологии             : {CLIMATOLOGY_ACC:.4f}")
print(f"раздувание от одного незаконного признака : {leaky_cv - safe_cv:+.4f} п.п. точности")
print(
    "\nМодель, использующая доходность будущего, нельзя торговать, нельзя выкатить, и при этом\n"
    "она показывает высочайшую точность в ноутбуке. Именно поэтому 'высокая точность' — не результат.\n"
    "Детектор выше — переносимый артефакт: ему не нужно знать, какой столбец незаконен,\n"
    "достаточно корреляции с целью."
)

топ-5 корреляций признак/цель ДО добавления утечки
  feature  max_biserial_r  vs_class  abs_r  ratio_to_median  flagged
range_pct         -0.1048         2 0.1048           3.1563    False
   rsi_14          0.0865         2 0.0865           2.6035    False
 ma_ratio          0.0598         2 0.0598           1.7994    False
   atr_14         -0.0528         2 0.0528           1.5891    False
ret_lag_3          0.0467         2 0.0467           1.4055    False

топ-5 корреляций признак/цель ПОСЛЕ добавления утечки
            feature  max_biserial_r  vs_class  abs_r  ratio_to_median  flagged
FORWARD_RETURN_LEAK          0.6448         0 0.6448          18.1293     True
          range_pct         -0.1048         2 0.1048           2.9476    False
             rsi_14          0.0865         2 0.0865           2.4313    False
           ma_ratio          0.0598         2 0.0598           1.6804    False
             atr_14         -0.0528         2 0.0528           1.4840    False

призн

# Эксперимент 4 — сколько в «лучшей модели» шума отбора?

Двадцать четыре конфигурации градиентного бустинга оцениваются очищенным CV на окне IS.
Конфигурация с лучшей CV-точностью — та, которую обычный рабочий процесс выкатил бы в прод.

Далее измеряются две вещи, **и ни одна из них рабочий процесс сам бы не отчитался:**

1. **Предсказывает ли CV-ранжирование OOS-ранжирование?** Корреляция Спирмена между двумя
   порядковыми шкалами показывает, сколько информации кросс-валидация реально перенесла.
2. **Обошло ли что-нибудь в сетке нулевой бенчмарк?** Порог — бенчмарк большинства классов из
   предыдущего раздела. Конфигурации, которые проходят его в CV, а затем не проходят вне
   выборки, — это **наблюдаемая напрямую цена тюнинга**; если не проходит ничего вообще, честный
   вывод — сетка нечего отбирать, что сильнее и менее лестно, чем переобучение. **Обе ветки
   печатаются.**

**Для контраста печатается третье число — buy-and-hold.** Классификатор, предсказывающий
направление, конкурирует с простым удержанием индекса, который рос, и бэктест без этого
сравнения не является бэктестом. Его Sharpe, Sortino и CAGR печатаются рядом со стратегией по
всему ходу.

**Финальное окно OOS здесь используется как *измерительный прибор* для смещения отбора.** Это
намеренное и задекларированное нарушение правила одного касания — без выделенного набора
смещение отбора измерить невозможно вовсе, — и именно поэтому ни один более поздний эксперимент
не выбирает по этим числам.

In [9]:
from itertools import product as _product

GRID = {
    "learning_rate": [0.03, 0.12],
    "max_leaf_nodes": [7, 15, 31],
    "min_samples_leaf": [20, 40],
    "l2_regularization": [0.0, 1.0],
    "max_iter": [150],
}
config_names = [dict(zip(GRID.keys(), combo))
                for combo in _product(*GRID.values())]
N_CONFIGS = len(config_names)
print(f"конфигураций к проверке: {N_CONFIGS}  <- это и есть n_trials для DSR ниже")


def make_hgb(cfg):
    return HistGradientBoostingClassifier(
        learning_rate=cfg["learning_rate"],
        max_leaf_nodes=cfg["max_leaf_nodes"],
        min_samples_leaf=cfg["min_samples_leaf"],
        l2_regularization=cfg["l2_regularization"],
        max_iter=cfg["max_iter"],
        random_state=RNG_SEED,
    )


sweep_rows = []
for i, cfg in enumerate(config_names):
    cv_acc = cv_mean_accuracy(purged_IS, X_IS, y_IS, lambda c=cfg: make_hgb(c))
    # Окно OOS здесь — измерительный прибор для смещения отбора, а не
    # поверхность отбора: по этим числам не выбирается ничего дальше.
    pre = FoldSafePreprocessor().fit(X_IS)
    model = make_hgb(cfg)
    model.fit(pre.transform(X_IS), y_IS.to_numpy())
    oos_acc = float((model.predict(pre.transform(X_OOS)) == y_OOS.to_numpy()).mean())
    sweep_rows.append({
        "config_id": i,
        "learning_rate": cfg["learning_rate"],
        "max_leaf_nodes": cfg["max_leaf_nodes"],
        "min_samples_leaf": cfg["min_samples_leaf"],
        "l2": cfg["l2_regularization"],
        "cv_accuracy": cv_acc,
        "oos_accuracy": oos_acc,
    })

sweep = pd.DataFrame(sweep_rows)
sweep["cv_rank"] = sweep["cv_accuracy"].rank(ascending=False, method="average").astype(int)
sweep["oos_rank"] = sweep["oos_accuracy"].rank(ascending=False, method="average").astype(int)

rho_cv_oos = float(stats.spearmanr(sweep["cv_rank"], sweep["oos_rank"]).statistic)
best_cfg_row = sweep.loc[sweep["cv_accuracy"].idxmax()]
worst_cfg_row = sweep.loc[sweep["cv_accuracy"].idxmin()]

print(f"\nлучшие 8 конфигураций по точности очищенного CV")
print(sweep.sort_values("cv_accuracy", ascending=False).head(8).round(4).to_string(index=False))
print(f"\nточность CV по всем {N_CONFIGS} конфигам : "
      f"{sweep['cv_accuracy'].min():.4f} -> {sweep['cv_accuracy'].max():.4f}")
print(f"ранговая корреляция Спирмена, CV-ранг к OOS-рангу : {rho_cv_oos:+.3f}")
print(f"лучшая по CV : конфигурация {int(best_cfg_row.config_id)}  "
      f"CV {best_cfg_row.cv_accuracy:.4f} -> OOS {best_cfg_row.oos_accuracy:.4f} "
      f"(ранг OOS {int(best_cfg_row.oos_rank)}/{N_CONFIGS})")
print(f"худшая по CV: конфигурация {int(worst_cfg_row.config_id)}  "
      f"CV {worst_cfg_row.cv_accuracy:.4f} -> OOS {worst_cfg_row.oos_accuracy:.4f} "
      f"(ранг OOS {int(worst_cfg_row.oos_rank)}/{N_CONFIGS})")

beat_cv = sweep[sweep["cv_accuracy"] > CLIMATOLOGY_ACC]
lucky = beat_cv[beat_cv["oos_accuracy"] <= CLIMATOLOGY_ACC]
below_clim_oos = int((sweep["oos_accuracy"] <= CLIMATOLOGY_ACC).sum())
cv_to_oos_drop = float(best_cfg_row["cv_accuracy"] - best_cfg_row["oos_accuracy"])
# Плюс означает, что кросс-валидация выбрала конфигурацию, которая перенеслась лучше, чем
# случайно выбранная; минус — что отбор не обошёл даже среднюю.
selected_minus_mean = float(best_cfg_row["oos_accuracy"] - sweep["oos_accuracy"].mean())

print(f"\nконфигураций, обогнавших климатологию в CV : {len(beat_cv)} / {N_CONFIGS}")
print(f"...из них НЕ смогли обойти её вне выборки : {len(lucky)} / {len(beat_cv)}")
print(f"конфигураций на уровне или ниже климатологии вне выборки : {below_clim_oos} / {N_CONFIGS}")
print(f"\nточность OOS лучшей по CV конфигурации : {best_cfg_row['oos_accuracy']:.4f}")
print(f"  её собственная точность CV           : {best_cfg_row['cv_accuracy']:.4f}")
print(f"  деградация от CV к OOS               : {cv_to_oos_drop:+.4f}")
print(f"диапазон точности OOS по всем конфигам : "
      f"{sweep['oos_accuracy'].min():.4f} -> {sweep['oos_accuracy'].max():.4f}")
print(f"средняя точность OOS по всем конфигам  : {sweep['oos_accuracy'].mean():.4f}")
print(f"лучшая по CV минус средняя, вне выборки: {selected_minus_mean:+.4f}")
if selected_minus_mean <= 0:
    print("  Минус: кросс-валидация не выбрала даже конфигурацию, обогнавшую случайно")
    print("  выбранную вне выборки. Отбор не перенёс сюда ничего.")
else:
    print("  Плюс, но небольшой: отбор перенёс немного, гораздо меньше, чем")
    print("  обещала внутри выборочная ранжировка.")
print(f"климатология для сравнения             : {CLIMATOLOGY_ACC:.4f}")

if len(beat_cv) == 0:
    print("\nВ сетке ничего не обогнало бенчмарк большинства классов — ни в CV, ни вне выборки.")
    print("Так что цена поиска по 24 конфигурациям не была уплачена ложными срабатываниями —")
    print("сигнала для поиска просто не было. Это другой и более")
    print("фундаментальный провал, чем переобучение, и это честный заголовок: на этих")
    print("признаках и этой цели градиентный бустинг не содержит пригодного")
    print("направленного преимущества над «всегда длинный SPY». Поэтому среднее по конфигурациям —")
    print("то число, которое нужно цитировать, и оно ниже бенчмарка.")
else:
    print("\nСреднее по всем конфигурациям, а не максимум, — честное ожидание для")
    print("конфигурации, взятой вслепую, без знания, какую любила CV. Читайте это число.")

конфигураций к проверке: 24  <- это и есть n_trials для DSR ниже



лучшие 8 конфигураций по точности очищенного CV
 config_id  learning_rate  max_leaf_nodes  min_samples_leaf  l2  cv_accuracy  oos_accuracy  cv_rank  oos_rank
         6           0.03              15                40 0.0       0.4617        0.4682        1         1
         7           0.03              15                40 1.0       0.4588        0.4519        2        12
         8           0.03              31                20 0.0       0.4588        0.4617        3         4
         9           0.03              31                20 1.0       0.4576        0.4535        4        10
         2           0.03               7                40 0.0       0.4572        0.4584        5         7
         3           0.03               7                40 1.0       0.4564        0.4519        6        12
        16           0.12              15                20 0.0       0.4560        0.4470        7        15
         4           0.03              15                20 0.0       0

# Эксперимент 4b — коррекция выбранного Sharpe за сам факт отбора

**В чём проблема.** Выбор лучшего из *N* попыток систематически завышает его Sharpe: удача
выглядит как навык. Это называется selection bias, и он есть у **любого** бэктеста, который
сообщил «лучшую» модель.

**Что считается.**
`probabilistic_sharpe_ratio` спрашивает, отличим ли реализованный Sharpe от нуля;
`deflated_sharpe_ratio` задаёт более трудный вопрос — отличим ли он от **лучшего из N попыток,
полученных чисто на удачу**. `min_track_record_length` отвечает, сколько наблюдений вообще
нужно, чтобы подтвердить целевой Sharpe.

Вводные устанавливаются здесь: выбранная конфигурация и **честное** количество попыток, её
породившее. Само вычисление выполняется в следующей ячейке на потоке чистых доходностей
walk-forward выбранной конфигурации.

In [10]:
SELECTED = config_names[int(best_cfg_row.config_id)]
print("конфигурация, которую выкатил бы обычный рабочий процесс, отобранная очищенным CV только на IS:")
print(json.dumps(SELECTED, indent=2))
print(f"\nчисло попыток для дефляции Sharpe: {N_CONFIGS}")
print("Именно пропуск этого аргумента заставляет оттюненный бэктест сообщать Sharpe без смысла.")

dsr_undeflated = deflated_sharpe_ratio(pd.Series([0.0, 1.0]), n_trials=1)
print("\nпроверка корректности реализации (вырожденный поток из 2 наблюдений):")
print(json.dumps({k: v for k, v in dsr_undeflated.items() if k in ("sr", "psr", "dsr")}, indent=2,
                 default=str))
print("Sharpe из двух наблюдений не несёт информации, как и должно быть.")

конфигурация, которую выкатил бы обычный рабочий процесс, отобранная очищенным CV только на IS:
{
  "learning_rate": 0.03,
  "max_leaf_nodes": 15,
  "min_samples_leaf": 40,
  "l2_regularization": 0.0,
  "max_iter": 150
}

число попыток для дефляции Sharpe: 24
Именно пропуск этого аргумента заставляет оттюненный бэктест сообщать Sharpe без смысла.

проверка корректности реализации (вырожденный поток из 2 наблюдений):
{
  "sr": 0.7071067811865475,
  "psr": NaN,
  "dsr": NaN
}
Sharpe из двух наблюдений не несёт информации, как и должно быть.


# Эксперимент 5 — зависит ли ответ от того, где стояло тестовое окно?

**В чём идея.** Один путь walk-forward — одна выборка из распределения возможных результатов.
Комбинаторный очищенный CV даёт их много: выборка режется на группы, и каждая комбинация
тестовых групп становится отдельным путём бэктеста.

**Что это даёт.** Разброс по путям отвечает на вопрос, на который ни один одиночный hold-out не
ответит: **сколько в этом результате свойства стратегии, а сколько — свойства тех дат, которые
я случайно выбрал?**

**Как считается.** Чистые доходности out-of-fold вычисляются один раз очищенным K-fold по всей
выборке и затем нарезаются на пути. Значения, передаваемые в `cpcv_backtest_paths`, — это
**чистые доходности за бар**, а не позиции, поэтому Sharpe каждого пути является настоящим
Sharpe настоящей кривой капитала.

In [11]:
n_groups = 6
n_test_groups = 2
cpcv_paths = combinatorial_purged_cv_splits(
    n_obs, n_groups=n_groups, n_test_groups=n_test_groups, embargo=EMBARGO, t1=HORIZON)
print(f"CPCV: {n_groups} групп, {n_test_groups} тестовых групп на путь -> "
      f"{len(cpcv_paths)} путей бэктеста")

# Вероятности вне фолда на всей выборке: каждый бар предсказан моделью, которая
# никогда его не видела, на очищенных фолдах с embargo.
present: set[int] = set()
oof_prob = np.full((n_obs, len(CLASS_ORDER)), np.nan)
for tr, te in purged_kfold_splits(n_obs, n_splits=n_groups, embargo=EMBARGO, t1=HORIZON):
    pre = FoldSafePreprocessor().fit(X.iloc[tr])
    model = make_hgb(SELECTED)
    model.fit(pre.transform(X.iloc[tr]), y.iloc[tr].to_numpy())
    present |= set(int(c) for c in model.classes_)
    # Сопоставляем столбцы классов этого фолда с глобальным порядком, чтобы фолды были сравнимы.
    proba = model.predict_proba(pre.transform(X.iloc[te]))
    for src_i, cls in enumerate(model.classes_):
        oof_prob[te, CLASS_ORDER.index(int(cls))] = proba[:, src_i]
    oof_prob[te] = np.nan_to_num(oof_prob[te], nan=0.0)

print(f"классы в каждом обучающем фолде: {sorted(present)} (ожидалось {CLASS_ORDER})")
covered = ~np.isnan(oof_prob).any(axis=1)
print(f"баров с предсказанием вне фолда     : {int(covered.sum())} / {n_obs}")

buy_i, sell_i = CLASS_ORDER.index(1), CLASS_ORDER.index(0)
idx_covered = np.flatnonzero(covered)
oof_pos = np.full(n_obs, np.nan)
oof_pos[covered] = apply_causal_execution(
    oof_prob[covered], buy=buy_i, sell=sell_i, confirm_bars=2, min_confidence=0.40)

valid = covered & np.isfinite(oof_pos)
idx_valid = np.flatnonzero(valid)
oof_net = np.full(n_obs, np.nan)
oof_net[idx_valid] = positions_to_returns(
    oof_pos[idx_valid], close.iloc[idx_valid], cost_bps=COST_BPS).to_numpy()

path_stats = cpcv_backtest_paths(np.nan_to_num(oof_net, nan=0.0), cpcv_paths)
path_sharpes = np.array(list(path_stats["path_sharpe"].values()))
print(f"\nSharpe по путям: среднее {path_stats['mean_path_sharpe']:.3f} | "
      f"p05 {path_stats['p05_path_sharpe']:.3f} | p95 {path_stats['p95_path_sharpe']:.3f}")
print(f"Один путь (среднее) отчитался бы одним из этих значений без всякого интервала.")
print(f"доля путей с Sharpe <= 0 : {(path_sharpes <= 0).mean():.1%}")
print(f"доля позиции OOS (баров в лонге): {np.nanmean(np.abs(oof_pos[valid])):.3f}")
print(f"оборот: {np.abs(np.diff(oof_pos[valid])).sum():.0f} смен позиции на "
      f"{int(valid.sum())} барах")
print("\nОтрицательный пятый процентиль означает, что Sharpe одного пути — небезопасная сводка")
print("распределения. Это одна выборка из него.")

CPCV: 6 групп, 2 тестовых групп на путь -> 15 путей бэктеста


классы в каждом обучающем фолде: [0, 1, 2] (ожидалось [0, 1, 2])
баров с предсказанием вне фолда     : 3065 / 3065

Sharpe по путям: среднее -0.365 | p05 -0.877 | p95 0.147
Один путь (среднее) отчитался бы одним из этих значений без всякого интервала.
доля путей с Sharpe <= 0 : 80.0%
доля позиции OOS (баров в лонге): 0.996
оборот: 516 смен позиции на 3065 барах

Отрицательный пятый процентиль означает, что Sharpe одного пути — небезопасная сводка
распределения. Это одна выборка из него.


## Разворачиваемая симуляция

Всё до этого было механизмом валидации. Эта ячейка производит тот один артефакт, который
исследовательской команде действительно нужен: **каузальную walk-forward симуляцию выбранной
конфигурации, за вычетом издержек, с приложенной неопределённостью.**

**Протокол, сформулированный точно:**

* Расширяющееся обучающее окно, переобучение каждые `REFIT_EVERY` баров.
* Каждый бар оценивается моделью, обученной **строго на более ранних барах**.
* Статистики предобработки пересчитываются внутри каждого переобучения.
* Сигналы проходят фильтр подтверждения, затем превращаются в позиции, затем в доходности с издержками.
* Ни один параметр нигде не выбирается по финальному окну в 20%.

In [12]:
wf_prob = np.full((n_obs, len(CLASS_ORDER)), np.nan)
refits = 0
for start in range(EMBARGO + n_oos, n_obs, REFIT_EVERY):
    end = min(start + REFIT_EVERY, n_obs)
    tr = np.arange(0, start)
    pre = FoldSafePreprocessor().fit(X.iloc[tr])
    model = make_hgb(SELECTED)
    model.fit(pre.transform(X.iloc[tr]), y.iloc[tr].to_numpy())
    proba = model.predict_proba(pre.transform(X.iloc[start:end]))
    # Сопоставляем столбцы классов этой модели с глобальным порядком.
    for src_i, cls in enumerate(model.classes_):
        wf_prob[start:end, CLASS_ORDER.index(int(cls))] = proba[:, src_i]
    wf_prob[start:end] = np.nan_to_num(wf_prob[start:end], nan=0.0)
    refits += 1

wf_valid = ~np.isnan(wf_prob).any(axis=1)
wf_idx = np.flatnonzero(wf_valid)
wf_prob_v = wf_prob[wf_idx]
wf_pos = apply_causal_execution(wf_prob_v, buy=buy_i, sell=sell_i,
                                confirm_bars=2, min_confidence=0.40)
prices_v = close.iloc[wf_idx]
net = positions_to_returns(wf_pos, prices_v, cost_bps=COST_BPS)
bh = buy_and_hold_returns(prices_v)

bench = pd.DataFrame({"strategy_net": net, "buy_and_hold": bh}).dropna()
net_s, bh_s = bench["strategy_net"], bench["buy_and_hold"]

# Walk-forward охватывает весь симулируемый период, включая отрезок IS.
# Отрезок IS — не чистое чтение (конфигурация отбиралась на нём), поэтому OOS
# выводится отдельно и именно это число цитируется.
oos_net = net_s.loc[net_s.index >= X_OOS.index[0]]
oos_bh = bh_s.loc[bh_s.index >= X_OOS.index[0]]


def max_drawdown(r: pd.Series) -> float:
    equity = (1.0 + r).cumprod()
    return float((equity / equity.cummax() - 1.0).min())


n_trades = int((np.abs(np.diff(wf_pos)) > 0).sum())
metrics = {
    "bars": int(len(net_s)),
    "trades": n_trades,
    "CAGR": annualised_return(net_s),
    "Sharpe": sharpe_ratio(net_s),
    "Sortino": sortino_ratio(net_s),
    "buy_hold_Sharpe": sharpe_ratio(bh_s),
    "buy_hold_CAGR": annualised_return(bh_s),
    "max_drawdown": max_drawdown(net_s),
    "profit_factor": profit_factor(net_s),
    "win_rate": win_rate(net_s),
}
oos_metrics = {
    "bars": int(len(oos_net)),
    "CAGR": annualised_return(oos_net),
    "Sharpe": sharpe_ratio(oos_net),
    "Sortino": sortino_ratio(oos_net),
    "buy_hold_Sharpe": sharpe_ratio(oos_bh),
    "buy_hold_CAGR": annualised_return(oos_bh),
    "max_drawdown": max_drawdown(oos_net),
}

print(f"walk-forward: {len(net_s)} баров | {refits} переобучений | {n_trades} сделок | "
      f"{COST_BPS:.0f} bp за полный круг")
print(f"период симуляции: {net_s.index[0].date()} -> {net_s.index[-1].date()}")
print("\nвесь период симуляции (включая отрезок IS, на котором отбиралась конфигурация)")
for k, v in metrics.items():
    print(f"  {k:<18}: {v:>12.4f}")
print(f"\nтолько отрезок OOS, {oos_net.index[0].date()} -> {oos_net.index[-1].date()} "
      "(никогда не использовался для отбора)")
for k, v in oos_metrics.items():
    print(f"  {k:<18}: {v:>12.4f}")

edge = oos_metrics["Sharpe"] - oos_metrics["buy_hold_Sharpe"]
print(f"\n  преимущество OOS Sharpe над buy-and-hold : {edge:+.4f}")
print("  Buy-and-hold приведён, потому что это бенчмарк, который действительно важен для")
print("  стратегии только в лонг, и именно его `HW1`-`HW3` никогда не сообщали.")

walk-forward: 2447 баров | 117 переобучений | 262 сделок | 10 bp за полный круг
период симуляции: 2017-05-17 -> 2026-10-01

весь период симуляции (включая отрезок IS, на котором отбиралась конфигурация)
  bars              :    2447.0000
  trades            :     262.0000
  CAGR              :      -0.0753
  Sharpe            :      -0.3409
  Sortino           :      -0.4324
  buy_hold_Sharpe   :       0.8444
  buy_hold_CAGR     :       0.1460
  max_drawdown      :      -0.5973
  profit_factor     :       0.9351
  win_rate          :       0.4736

только отрезок OOS, 2024-05-28 -> 2026-10-01 (никогда не использовался для отбора)
  bars              :     613.0000
  CAGR              :      -0.0586
  Sharpe            :      -0.3035
  Sortino           :      -0.4100
  buy_hold_Sharpe   :       1.1060
  buy_hold_CAGR     :       0.1767
  max_drawdown      :      -0.2814

  преимущество OOS Sharpe над buy-and-hold : -1.4095
  Buy-and-hold приведён, потому что это бенчмарк, который действ

In [13]:
ci = bootstrap_sharpe_ci(net_s, n_boot=4000, block_size=10, alpha=0.05, seed=RNG_SEED)
psr = probabilistic_sharpe_ratio(net_s, sr_benchmark=0.0)
dsr = deflated_sharpe_ratio(net_s, n_trials=N_CONFIGS)
mintrl_bars = min_track_record_length(net_s, sr_target=SR_TARGET)
mintrl_bh = min_track_record_length(bh_s, sr_target=SR_TARGET)
MINTRL_DEFINED = bool(np.isfinite(mintrl_bars))

print("неопределённость Sharpe в walk-forward")
print(f"  точечная оценка           : {ci['point']:.3f}")
print(f"  интервал 95% блочного бутстрэпа : [{ci['lo']:.3f}, {ci['hi']:.3f}]  ({ci['n_boot']} выборок)")
print(f"  P(Sharpe > 0)             : {ci['p_gt_zero']:.3f}")
print(f"  PSR против нуля           : {psr:.4f}")
print(f"  DSR против лучшего из {N_CONFIGS} попыток : {dsr['dsr']:.4f} "
      f"(бенчмарк SR {dsr['sr_benchmark']:.3f})")
if MINTRL_DEFINED:
    print(f"  MinTRL для SR {SR_TARGET:.1f}          : {mintrl_bars:,.0f} баров "
      f"({mintrl_bars/TRADING_DAYS:.1f} лет) против {len(net_s)} наблюдённых")
else:
    print(f"  MinTRL для SR {SR_TARGET:.1f}          : не определён")
    print("    MinTRL не определён, когда реализованный Sharpe не положителен: никакая длина")
    print("    этого потока доходностей не подтвердит целевой Sharpe, потому что положительного")
    print("    Sharpe нет, чтобы его подтвердить. Это правильный ответ, а не пропуск.")
print(f"  для контраста MinTRL buy-and-hold при SR {SR_TARGET:.1f}: "
      f"{mintrl_bh/TRADING_DAYS:.1f} лет (его Sharpe положителен)")

CI_EXCLUDES_ZERO = bool(np.isfinite(ci["lo"]) and ci["lo"] > 0)
DSR_CREDIBLE = bool(np.isfinite(dsr["dsr"]) and dsr["dsr"] >= 0.95)
VERDICT = ("достоверное преимущество" if (CI_EXCLUDES_ZERO and DSR_CREDIBLE)
           else "преимущества нет после коррекции за отбор")

print(f"\nИнтервал 95% не включает ноль : {CI_EXCLUDES_ZERO}")
print(f"DSR >= 0.95          : {DSR_CREDIBLE}")
print(f"ВЕРДИКТ              : {VERDICT}")
if not CI_EXCLUDES_ZERO:
    print("\nИнтервал включает ноль. Точечная оценка неотличима от удачи на объёме")
    print("этой выборки, и никакие ухищрения с моделью не изменят этого факта — только")
    print("более длинная история торговли.")
if not DSR_CREDIBLE:
    print(f"\nПосле дефляции за {N_CONFIGS} испробованных конфигураций Sharpe не")
    print("отличим от лучшего из такого числа удачных догадок.")
if VERDICT == "credible edge":
    print("\nИнтервал исключает ноль И переживает коррекцию за отбор. Это тот порог,")
    print("который не проходят большинство опубликованных бэктестов.")
if MINTRL_DEFINED:
    print(f"\nMinTRL говорит: нужно {mintrl_bars/TRADING_DAYS:.1f} лет данных, чтобы подтвердить")
    print(f"вообще Sharpe {SR_TARGET:.1f}. В этом запуске {len(net_s)/TRADING_DAYS:.1f} лет.")
else:
    print(f"\nBuy-and-hold нужно {mintrl_bh/TRADING_DAYS:.1f} лет, чтобы подтвердить SR "
      f"{SR_TARGET:.1f}; этой стратегии\nнужна неограниченная история, потому что её "
      f"реализованный Sharpe отрицателен. В этой асимметрии и есть результат.")

неопределённость Sharpe в walk-forward
  точечная оценка           : -0.341
  интервал 95% блочного бутстрэпа : [-0.899, 0.218]  (4000 выборок)
  P(Sharpe > 0)             : 0.120
  PSR против нуля           : 0.1441
  DSR против лучшего из 24 попыток : 0.0016 (бенчмарк SR 0.038)
  MinTRL для SR 1.0          : не определён
    MinTRL не определён, когда реализованный Sharpe не положителен: никакая длина
    этого потока доходностей не подтвердит целевой Sharpe, потому что положительного
    Sharpe нет, чтобы его подтвердить. Это правильный ответ, а не пропуск.
  для контраста MinTRL buy-and-hold при SR 1.0: 1.4 лет (его Sharpe положителен)

Интервал 95% не включает ноль : False
DSR >= 0.95          : False
ВЕРДИКТ              : преимущества нет после коррекции за отбор

Интервал включает ноль. Точечная оценка неотличима от удачи на объёме
этой выборки, и никакие ухищрения с моделью не изменят этого факта — только
более длинная история торговли.

После дефляции за 24 испробованных конф

## Выводы

Числа выше печатаются кодом и вычислены в этом запуске; рассуждение сформулировано отдельно,
чтобы можно было сверить одно с другим.

**1. Дисциплина валидации — это не бухгалтерия, а результат.** Три правила разбиения, применённые
к одной модели и одному набору признаков, выбрали три разные модели и разошлись в точности вне
выборки ровно на величину, напечатанную в эксперименте 1. Там, где разброс CV по всей сетке
сопоставим с расхождением между схемами, сетка не разрешается данными, и выбор максимума — это
выбор шума.

**2. Два дешёвых бага утечки повели себя по-разному, и отчёт о обоих — и есть цель.** Скейлер,
обученный на train+test, искажает CV-точность на величину из эксперимента 2 — на этом наборе
признаков незначительно, потому что большинство столбцов уже являются отношениями и
ограниченными осцилляторами. Один признак, смотрящий вперёд, раздувает точность примерно на
порядок сильнее, в том же направлении. **Переносимая привычка — это аудит, а не интуиция:**
«мои признаки, наверное, причинны» — не контроль, контроль — это корреляция с целью. Обратите
внимание: корреляция утечки заметно ниже 1.0, потому что deadband создаёт средний класс, так что
абсолютный порог бы её пропустил — **детектор должен быть относительным к остальному набору
признаков.**

**3. Тюнинг — это стоимость, и на этой задаче он не нашёл вообще ничего.** Перебор в эксперименте
4 запускает 24 конфигурации. Честное прочтение зависит от того, что показывает запуск, и **обе
ветки печатаются**: если конфигурации бьют бенчмарк большинства в CV, а затем проваливаются
вне выборки, это количество — цена поиска; если бенчмарк не бьёт ничего нигде, цена не была
уплачена, потому что сигнала не было найти, что результат фундаментальнее переобучения. В любом
случае цитировать нужно **среднюю** точность по конфигурациям, а не максимум, и читать её рядом
с бенчмарком большинства классов. **Бэктест на шести сделках из `HW2` — крайняя версия этого
провала:** максимум по неуказанному числу попыток, сообщённый так, будто это была единственная
попытка.

**4. Один путь бэктеста — одна выборка из распределения.** Ячейка CPCV печатает пятый и 95-й
процентили распределения Sharpe путей. Когда пятый процентиль отрицателен, цитировать Sharpe
одного пути как *тот самый* Sharpe — ошибка категории, и это относится к **каждому** бэктесту
в этом репозитории, сообщавшему одно число.

**5. Buy-and-hold — тот бенчмарк, которого не хватало.** Преимущество над константой —
арифметическое курьёз; преимущество над нулевым buy-and-hold бенчмарком — единственный вид,
который приносит деньги. Это сравнение находится в ячейке walk-forward, рядом с интервалом
неопределённости.

**Что наследует `HW8`.** Зооподбор моделей, оцениваемый против нулевых бейзлайнов; счётчик
отбора для дефляции Sharpe; очищенная с embargo кросс-валидация; издержки в потоке
доходностей; интервал неопределённости, приложенный к каждой заглавной цифре.

In [14]:
print("=" * 78)
print("ГЛАВНЫЙ ВЫВОД, посчитан в этом запуске")
print("=" * 78)

print("\nДанные и протокол")
print(f"  тикер / источник             : {TICKER} / {source}")
print(f"  баров (признаков)            : {n_obs}")
print(f"  IS (только для отбора)       : {len(X_IS)}")
print(f"  OOS (финальное, один раз)    : {len(X_OOS)}")
print(f"  точность климатологии        : {CLIMATOLOGY_ACC:.4f} "
      f"(всегда '{CLASS_NAMES[MAJORITY_CLASS]}')")

print("\n1. Меняет ли правило разбиения ответ?")
print(f"  разброс точности OOS по 3 схемам : {oos_spread:.4f}")
print(f"  разброс точности CV по 6 значениям C: {cv_spread:.4f}")
print(f"  конфигурация, выбранная каждой схемой : "
      + " | ".join(f"{r['scheme']} -> C={r['selected_C']:g}" for _, r in oos_tbl.iterrows()))
print(f"  схемы согласны в выборе модели       : "
      f"{oos_tbl['selected_C'].nunique() == 1}")

print("\n2. Утечка в предобработке (скейлер на train+test)")
print(f"  макс. искажение точности CV   : {max_cv_delta:+.4f}")
print(f"  макс. искажение точности OOS  : {max_oos_delta:+.4f}")
print(f"  значимо ( > 0.005 )           : {LEAK_EFFECT_MATERIAL}")

print("\n3. Признак, смотрящий вперёд")
print(f"  точность CV, причинные        : {safe_cv:.4f}")
print(f"  точность CV, +1 незаконный     : {leaky_cv:.4f}")
print(f"  раздувание                     : {leaky_cv - safe_cv:+.4f}")
print(f"  |r| утечки против медианы |r|   : {leak_row['abs_r']:.3f} против "
      f"{audit_after.attrs['median_abs_r']:.3f} ({leak_row['ratio_to_median']:.1f} раза)")
print(f"  отмечено автоматически аудитом : {len(flagged)} признак(ов)")
if len(flagged):
    print(f"    -> {', '.join(flagged['feature'])}")

print("\n4. Шум отбора")
print(f"  испробовано конфигураций      : {N_CONFIGS}")
print(f"  Спирмен(CV-ранг, OOS-ранг)     : {rho_cv_oos:+.3f}")
print(f"  обогнали климатологию в CV    : {len(beat_cv)}")
print(f"  ...провалились вне выборки    : {len(lucky)}")
print(f"  на уровне/ниже климатологии OOS: {below_clim_oos} / {N_CONFIGS}")
print(f"  средняя точность OOS (все конфиги): {sweep['oos_accuracy'].mean():.4f}")
print(f"  точность OOS лучшей по CV      : {best_cfg_row['oos_accuracy']:.4f} "
      f"(CV показывала {best_cfg_row['cv_accuracy']:.4f}, падение {cv_to_oos_drop:+.4f})")
print(f"  лучшая по CV минус средняя, точность OOS: {selected_minus_mean:+.4f}")
print(f"  что-нибудь обошло бенчмарк?   : {len(beat_cv) > 0}")

print("\n5. Распределение путей CPCV")
print(f"  путей                          : {path_stats['n_paths']}")
print(f"  Sharpe по путям p05 / среднее / p95: {path_stats['p05_path_sharpe']:.3f} / "
      f"{path_stats['mean_path_sharpe']:.3f} / {path_stats['p95_path_sharpe']:.3f}")
print(f"  доля путей с Sharpe <= 0       : {(path_sharpes <= 0).mean():.1%}")

print("\nСимуляция walk-forward, за вычетом издержек")
print(f"  баров / сделок / издержка      : {len(net_s)} / {n_trades} / {COST_BPS:.0f} bp")
print(f"  CAGR / Sharpe за весь период   : {metrics['CAGR']:+.4f} / {metrics['Sharpe']:.3f}")
print(f"  Sortino / макс. просадка за весь период: {metrics['Sortino']:.3f} / "
      f"{metrics['max_drawdown']:+.4f}")
print(f"  profit factor / доля побед за весь период: {metrics['profit_factor']:.3f} / "
      f"{metrics['win_rate']:.4f}")
print("  только отрезок OOS (никогда не использовался для отбора):")
print(f"    CAGR / Sharpe               : {oos_metrics['CAGR']:+.4f} / "
      f"{oos_metrics['Sharpe']:.3f}")
print(f"    CAGR / Sharpe buy-and-hold  : {oos_metrics['buy_hold_CAGR']:+.4f} / "
      f"{oos_metrics['buy_hold_Sharpe']:.3f}")
print(f"    преимущество Sharpe над buy-and-hold: {edge:+.4f}")
print(f"    Sortino / макс. просадка    : {oos_metrics['Sortino']:.3f} / "
      f"{oos_metrics['max_drawdown']:+.4f}")

print("\nUncertainty")
print(f"  интервал Sharpe 95% бутстрэпа : [{ci['lo']:.3f}, {ci['hi']:.3f}]")
print(f"  PSR vs zero                   : {psr:.4f}")
print(f"  DSR против лучшего из {N_CONFIGS}   : {dsr['dsr']:.4f}")
if MINTRL_DEFINED:
    print(f"  MinTRL для SR {SR_TARGET:.1f}          : {mintrl_bars/TRADING_DAYS:.1f} лет "
      f"(есть {len(net_s)/TRADING_DAYS:.1f})")
else:
    print(f"  MinTRL для SR {SR_TARGET:.1f}       : не определён (реализованный Sharpe <= 0)")
print(f"  MinTRL для buy-and-hold, SR {SR_TARGET:.1f}: {mintrl_bh/TRADING_DAYS:.1f} лет")
print(f"\n  ВЕРДИКТ                      : {VERDICT}")

print("\nЧто наследует HW8: нулевые бейзлайны (климатология И buy-and-hold), очищенная")
print("кросс-валидация с embargo, число конфигураций для DSR, издержки в потоке")
print("доходностей и интервал на каждое заглавное число.")

ГЛАВНЫЙ ВЫВОД, посчитан в этом запуске

Данные и протокол
  тикер / источник             : SPY / SPY (local: C:\Users\MV\AppData\Local\Temp\opencode\ML_Fin_Notebooks\data\SPY.csv)
  баров (признаков)            : 3065
  IS (только для отбора)       : 2452
  OOS (финальное, один раз)    : 613
  точность климатологии        : 0.5073 (всегда 'buy')

1. Меняет ли правило разбиения ответ?
  разброс точности OOS по 3 схемам : 0.0016
  разброс точности CV по 6 значениям C: 0.0229
  конфигурация, выбранная каждой схемой : случайный K-fold -> C=0.01 | очищенный K-fold -> C=0.01 | walk-forward -> C=0.1
  схемы согласны в выборе модели       : False

2. Утечка в предобработке (скейлер на train+test)
  макс. искажение точности CV   : +0.0020
  макс. искажение точности OOS  : +0.0033
  значимо ( > 0.005 )           : False

3. Признак, смотрящий вперёд
  точность CV, причинные        : 0.4829
  точность CV, +1 незаконный     : 0.9566
  раздувание                     : +0.4737
  |r| утечки против ме

In [15]:
summary = {
    "run": {
        "ticker": TICKER,
        "data_source": str(source),
        "n_bars": int(n_obs),
        "n_features": int(X.shape[1]),
        "is_bars": int(len(X_IS)),
        "oos_bars": int(len(X_OOS)),
        "horizon": HORIZON,
        "dead_band": DEAD_BAND,
        "embargo": EMBARGO,
        "cost_bps": COST_BPS,
        "refit_every": REFIT_EVERY,
    },
    "benchmark_climatology_accuracy": CLIMATOLOGY_ACC,
    "exp1_split_rule": {
        "oos_accuracy_by_scheme": oos_tbl.set_index("scheme")["oos_accuracy"].round(4).to_dict(),
        "selected_C_by_scheme": oos_tbl.set_index("scheme")["selected_C"].to_dict(),
        "oos_accuracy_spread": oos_spread,
        "cv_accuracy_spread_across_grid": cv_spread,
        "schemes_selected_same_model": bool(oos_tbl["selected_C"].nunique() == 1),
    },
    "exp2_preprocessing_leakage": {
        "max_cv_distortion": max_cv_delta,
        "max_oos_distortion": max_oos_delta,
        "material": bool(LEAK_EFFECT_MATERIAL),
    },
    "exp3_forward_feature": {
        "cv_accuracy_causal_only": safe_cv,
        "cv_accuracy_with_leak": leaky_cv,
        "inflation": leaky_cv - safe_cv,
        "leak_abs_biserial_r": float(leak_row["abs_r"]),
        "median_abs_biserial_r": audit_after.attrs["median_abs_r"],
        "ratio_to_median": float(leak_row["ratio_to_median"]),
        "features_flagged_by_audit": flagged["feature"].tolist(),
        "audit_detects_leak_automatically": bool(len(flagged) > 0),
        "abs_threshold_0_95_would_have_missed_it": bool(
            leak_row["abs_r"] < 0.95 and leak_row["ratio_to_median"] >= 5.0),
    },
    "exp4_selection_noise": {
        "n_configs": N_CONFIGS,
        "spearman_cv_rank_vs_oos_rank": rho_cv_oos,
        "beat_climatology_in_cv": int(len(beat_cv)),
        "failed_out_of_sample": int(len(lucky)),
        "at_or_below_climatology_oos": int(below_clim_oos),
        "mean_oos_accuracy_all_configs": float(sweep["oos_accuracy"].mean()),
        "oos_accuracy_cv_best": float(best_cfg_row["oos_accuracy"]),
        "cv_accuracy_of_cv_best": float(best_cfg_row["cv_accuracy"]),
        "cv_to_oos_degradation": cv_to_oos_drop,
        "cv_best_minus_mean_oos": selected_minus_mean,
        "anything_beat_the_benchmark": bool(len(beat_cv) > 0),
    },
    "exp5_cpcv": {
        "n_paths": int(path_stats["n_paths"]),
        "p05_path_sharpe": path_stats["p05_path_sharpe"],
        "mean_path_sharpe": path_stats["mean_path_sharpe"],
        "p95_path_sharpe": path_stats["p95_path_sharpe"],
        "share_paths_sharpe_le_zero": float((path_sharpes <= 0).mean()),
    },
    "walk_forward_full_span": {k: float(v) for k, v in metrics.items()},
    "walk_forward_oos_segment": {k: float(v) for k, v in oos_metrics.items()},
    "oos_segment_sharpe_edge_vs_buy_hold": edge,
    "uncertainty": {
        "sharpe_ci_lo": ci["lo"],
        "sharpe_ci_hi": ci["hi"],
        "p_sharpe_gt_zero": ci["p_gt_zero"],
        "psr": psr,
        "dsr": dsr["dsr"],
        "dsr_sr_benchmark": dsr["sr_benchmark"],
        "mintrl_years_for_target": (mintrl_bars / TRADING_DAYS) if MINTRL_DEFINED else None,
        "mintrl_defined": MINTRL_DEFINED,
        "mintrl_years_for_target_buy_hold": mintrl_bh / TRADING_DAYS,
        "observed_years": len(net_s) / TRADING_DAYS,
        "ci_excludes_zero": CI_EXCLUDES_ZERO,
        "dsr_credible": DSR_CREDIBLE,
        "verdict": VERDICT,
    },
}
print(json.dumps(summary, indent=2, default=str))

{
  "run": {
    "ticker": "SPY",
    "data_source": "SPY (local: C:\\Users\\MV\\AppData\\Local\\Temp\\opencode\\ML_Fin_Notebooks\\data\\SPY.csv)",
    "n_bars": 3065,
    "n_features": 23,
    "is_bars": 2452,
    "oos_bars": 613,
    "horizon": 1,
    "dead_band": 0.0005,
    "embargo": 5,
    "cost_bps": 10.0,
    "refit_every": 21
  },
  "benchmark_climatology_accuracy": 0.5073409461663948,
  "exp1_split_rule": {
    "oos_accuracy_by_scheme": {
      "\u0441\u043b\u0443\u0447\u0430\u0439\u043d\u044b\u0439 K-fold": 0.491,
      "\u043e\u0447\u0438\u0449\u0435\u043d\u043d\u044b\u0439 K-fold": 0.491,
      "walk-forward": 0.4894
    },
    "selected_C_by_scheme": {
      "\u0441\u043b\u0443\u0447\u0430\u0439\u043d\u044b\u0439 K-fold": 0.01,
      "\u043e\u0447\u0438\u0449\u0435\u043d\u043d\u044b\u0439 K-fold": 0.01,
      "walk-forward": 0.1
    },
    "oos_accuracy_spread": 0.0016313213703099683,
    "cv_accuracy_spread_across_grid": 0.02286050415915941,
    "schemes_selected_same_mo